# Urban sentiment: a review-to-city scorecard

**Objective:** use sampled Yelp reviews to build an Indianapolis scorecard that relates raw review text and star ratings to model-generated polarity, emotion, keywords, and location.

The default path reads a small cached structured subset and makes no download or API call. A full Yelp path is available only after the user places the licensed/terms-compliant JSON files in `data/`. Model-generated labels and sampled reviews do not establish population sentiment.

**Modular mapping:** review text + stars + coordinates → structured labels → city scorecard, distribution, and map.

**Prerequisites:** JSON-lines data, pandas groupby, and basic Folium/Matplotlib visualisation.

## 1. Load the sample and inspect its schema

The sample manifest records that these 60 rows are a deterministic subset of the tracked 300-row Indianapolis output. The output is cached model-labelled data, not human ground truth. The `full` branch uses `llm_utils.load_reviews_by_city` after the user supplies the Yelp JSON files.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from ccai9012 import llm_utils, viz_utils
from ccai9012.paths import STARTER_KITS_DIR, ensure_output_dir

DATA_SOURCE = "sample"  # "sample" is offline; "full" is opt-in.
CITY_NAME = "Indianapolis"
RUN_LIVE_LLM = False
MODULE_ROOT = STARTER_KITS_DIR / "2_llm_structure_output"
EXAMPLE_ROOT = MODULE_ROOT / "urban_sentiment"
OUTPUT_DIR = ensure_output_dir(EXAMPLE_ROOT)
manifest = json.loads((MODULE_ROOT / "sample_manifest.json").read_text(encoding="utf-8"))["urban_sentiment"]

if DATA_SOURCE == "sample":
    reviews_df = pd.read_csv(EXAMPLE_ROOT / "data/indy_reviews_sample.csv")
elif DATA_SOURCE == "full":
    business_path = EXAMPLE_ROOT / "data/yelp_academic_dataset_business.json"
    review_path = EXAMPLE_ROOT / "data/yelp_academic_dataset_review.json"
    if not business_path.is_file() or not review_path.is_file():
        raise FileNotFoundError("Download the Yelp JSON files into urban_sentiment/data before selecting full mode.")
    reviews = llm_utils.load_reviews_by_city(business_path, review_path, CITY_NAME, max_reviews=1000)
    reviews_df = pd.DataFrame(reviews).rename(columns={"text": "review_text"})
else:
    raise ValueError("DATA_SOURCE must be 'sample' or 'full'")

required = set(manifest["schema"])
assert required <= set(reviews_df.columns)
print(f"mode={DATA_SOURCE!r}; city={CITY_NAME}; rows={len(reviews_df)}")
display(reviews_df.head(3))

## 2. Preserve the raw answer before aggregating

A label is an interpretation layered on top of a review. Keep `review_text`, `stars`, coordinates, `polarity`, `sentiment`, and `keywords` together. The scorecard below can then show agreement or disagreement between stars and model labels without pretending either is a direct population measure.

![Sampled Yelp reviews retain raw text and ratings, receive cached or LLM labels, and become a cautious city scorecard.](../../../docs/figs/urban_sentiment_scorecard.svg)

In [ ]:
scorecard = (
    reviews_df.groupby("polarity", dropna=False)
    .agg(review_count=("review_text", "size"), mean_stars=("stars", "mean"))
    .reset_index()
)
scorecard["share"] = scorecard["review_count"] / scorecard["review_count"].sum()
scorecard.to_csv(OUTPUT_DIR / "offline_city_scorecard.csv", index=False)
display(scorecard)

plt.figure(figsize=(7, 4))
reviews_df.groupby(["stars", "polarity"]).size().unstack(fill_value=0).plot.bar(ax=plt.gca())
plt.title("Sampled labels by star rating")
plt.xlabel("Star rating")
plt.ylabel("Review count")
plt.tight_layout()
plt.show()

## 3. Map the sample, then ask what it cannot tell us

`viz_utils.plot_review_heatmap` expects a list of records with latitude and longitude and returns an interactive density map. `viz_utils.plot_review_map` accepts a DataFrame and colors points by a categorical field. These visuals show spatial concentration among sampled businesses; they do not show causal effects, representativeness, or sentiment of residents who did not leave reviews.

In [ ]:
display(viz_utils.plot_review_heatmap(reviews_df.to_dict("records"), zoom_start=11))
display(viz_utils.plot_review_map(
    reviews_df,
    name_field="business_name",
    sentiment_field="polarity",
    map_center=[reviews_df["latitude"].mean(), reviews_df["longitude"].mean()],
    zoom=11,
))

## 4. Explore keywords without treating them as prevalence

The word cloud is a compact inspection device: larger words occur more often in the sample's stored keyword strings. It is not a statistical estimate of which topics matter to all Indianapolis visitors or residents.

In [ ]:
viz_utils.plot_wordclouds(reviews_df, target_col="polarity", tag_col="keywords")
plt.show()

## Optional live lab and limitations

To label new raw Yelp reviews, set `RUN_LIVE_LLM = True`, call `llm_utils.analyze_reviews`, and save the raw response alongside the prompt/model/date metadata. The default notebook deliberately uses cached labels. For a stronger study, use a human-checked sample, report sampling rules and missingness, compare alternative labelers, and separate review-site activity from population sentiment.

In [ ]:
if RUN_LIVE_LLM:
    raise RuntimeError("Owner-approved live LLM check required; keep model and cost metadata with the output.")
print("offline_mode=True; cached labels retained; no Yelp download or LLM call was made")